# Solution procedure of the cohesive zone: consistent tangent, line search and bisection

The residual of the DCB with the `'SB_TSL'` connection is continuous but only piecewise smooth: its derivative jumps whenever an integration point crosses the separation at damage onset or at failure, or switches between loading and unloading (see the section *Solution procedure* of [Cohesive zone model](../cohesive_zone.rst)). The drivers therefore use three ingredients:

- the consistent tangent of the cohesive zone, the secant stiffness plus the damage-rate term of [`MultiDomain.calc_kT_TSL`](../multidomain_overview.rst);
- a backtracking line search on the scaled residual norm, `line_search_max` halvings of the step;
- the bisection of increments that do not converge, `max_bisections` levels.

This notebook solves the reference DCB of the [convergence study](dcb_mode_I_convergence_study.ipynb), case A ($k_o = 10^5$ N/mm$^3$), with parts of the procedure switched off, as the script `dcb_mode_I_solver_study.py` of this folder does:

| run | discretisation | consistent tangent | line search | bisection | iterations per opening |
|---|---|---|---|---|---|
| `ref` | $m_{tsl} = 25$, $n_x = 180$ | yes | 6 | 6 | 50 |
| `secant` | $m_{tsl} = 25$, $n_x = 180$ | no | 6 | no | 200 |
| `nols` | $m_{tsl} = 25$, $n_x = 180$ | yes | no | no | 200 |
| `nols_nx60` | $m_{tsl} = 25$, $n_x = 60$ | yes | no | no | 200 |
| `ls_nx60` | $m_{tsl} = 25$, $n_x = 60$ | yes | 6 | no | 200 |
| `ref_nx60` | $m_{tsl} = 25$, $n_x = 60$ | yes | 6 | 6 | 50 |

The runs without bisection stop at the first opening that does not converge. With $n_x = 60$ there are 3.6 integration points per cohesive zone length, too few for the rules of the convergence study. Every Newton-Raphson iteration is traced as (opening, iteration, criterion, step length).

**References**

- N. D'Souza, *Multi-Domain Semi-Analytical Cohesive Zone Approach*, MSc thesis, Delft University of Technology, 2024, https://resolver.tudelft.nl/uuid:4709f888-c032-4d9f-97e3-402dfdd47cf1.
- B. H. A. H. Tijs, S. Abdel-Monsef, J. Renart, A. Turon and C. Bisagni, *Characterization and analysis of the interlaminar behavior of thermoplastic composites considering fiber bridging and R-curve effects*, Composites Part A 162 (2022) 107101, https://doi.org/10.1016/j.compositesa.2022.107101.

In [ ]:
%matplotlib inline
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

sys.path.insert(0, os.path.abspath('.'))
import dcb_utils as du
import dcb_mode_I_convergence_study as cs
import dcb_mode_I_solver_study as ss

RUNS = dict(ss.RUNS)
RUNS['ref_nx60'] = dict()
BASE = dict(ss.BASE, ref_nx60='A_m25_nx60')
print(RUNS)

def md_table(header, rows):
    """Markdown table from a header and rows of already formatted cells"""
    lines = ['| ' + ' | '.join(header) + ' |', '|' + '---|'*len(header)]
    lines += ['| ' + ' | '.join(str(c) for c in r) + ' |' for r in rows]
    return Markdown('\n'.join(lines))


## Runs

Each run builds the case of the convergence study, `cs.case_dict`, with its openings, `cs.openings`, and calls `dcb_utils.solve_dcb` with the options of the table above.

In [ ]:
results, traces = {}, {}
for tag, opts in RUNS.items():
    p = cs.cases()[BASE.get(tag, 'A_m25_nx180')]
    trace = []
    results[tag] = du.solve_dcb(cs.case_dict(p), cs.openings(p['dw']), trace=trace, **opts)
    traces[tag] = np.array(trace)
    r = results[tag]
    print(f"{tag}: {len(r['delta'])} openings converged, last {r['delta'][-1]:.2f} mm, "
          f"{int(np.sum(r['iters']))} iterations, wall time {r['time']/60:.1f} min")

## Iterations

Openings converged, total iterations, iterations up to the peak and largest number per opening, bisected increments and the number of iterations whose line search reduced the step, $s < 1$.

In [ ]:
rows = []
for tag, r in results.items():
    it = np.asarray(r['iters'])
    ip = int(np.argmax(r['P']))
    tr = traces[tag]
    rows.append([tag, len(r['delta']), f"{r['delta'][-1]:.1f}", 'yes' if r['aborted'] else 'no',
                 int(it.sum()), int(it[:ip + 1].sum()), int(it.max()), int(r['failed']),
                 int(np.sum(tr[:, 3] < 1)), f"{r['P'][ip]:.2f}", f"{r['delta'][ip]:.1f}"])
display(md_table(['run', 'openings', 'last $\\delta$ (mm)', 'stopped', 'iterations', 'up to the peak',
                  'max per opening', 'bisections', 'reduced steps', '$P_{max}$ (N)',
                  '$\\delta(P_{max})$ (mm)'], rows))

The converged loads do not depend on the path to them: the loads of the runs that reach the same openings agree with those of `ref` to the tolerance of the convergence criterion. Largest difference of the load to `ref` over the common openings:

In [ ]:
for ref, tags in (('ref', ('secant', 'nols')), ('ref_nx60', ('nols_nx60', 'ls_nx60'))):
    r0 = results[ref]
    for tag in tags:
        r = results[tag]
        n = min(len(r['delta']), len(r0['delta']))
        d = np.max(np.abs(np.asarray(r['P'][:n]) - np.asarray(r0['P'][:n])))
        print(f'{tag} vs {ref}: {n} common openings, max |P - P_ref| = {d:.1e} N')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for tag in ('ref', 'secant', 'nols'):
    r = results[tag]
    ax.semilogy(r['delta'], r['iters'], '-o', ms=3, label=tag)
ax.axvline(results['ref']['delta'][int(np.argmax(results['ref']['P']))], color='0.5', lw=0.8)
ax.set_xlabel('opening $\\delta$ (mm)')
ax.set_ylabel('iterations per opening')
ax.set_title('$n_x$ = 180; vertical line: peak load')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.show()

## Cycle of the iterations without safeguards

At the last opening tried by the runs that stop, the convergence criterion of the last iterations. A cycle is detected when the last values repeat with a period of at most 12 iterations to a relative $10^{-9}$, `dcb_mode_I_solver_study.cycle`; the tangent stiffness of the panels is refreshed every $n_{NR} = 3$ iterations.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.2))
for tag in ('secant', 'nols_nx60', 'ls_nx60'):
    tr = traces[tag]
    last = tr[np.isclose(tr[:, 0], tr[-1, 0])]
    period, _ = ss.cycle(last[:, 2])
    stopped = results[tag]['aborted']
    print(f"{tag}: last opening tried {tr[-1, 0]:.2f} mm, {len(last)} iterations, stopped: {stopped}, "
          f"last criterion {last[-1, 2]:.3g}, cycle period: {period}")
    ax.semilogy(last[:, 1], last[:, 2], lw=1, label=f'{tag}, $\\delta$ = {tr[-1, 0]:.2f} mm')
ax.set_xlabel('iteration')
ax.set_ylabel('convergence criterion')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=8)
plt.show()

## Discussion

- **Consistent tangent.** On the converged discretisation the consistent tangent alone is enough: without line search and without bisection (`nols`) every opening converges in a few iterations, as with the full procedure (`ref`). With the secant stiffness as the only tangent of the interface (`secant`), the iterations grow as the process zone develops, and after the peak the criterion decreases only linearly, so the run stops at the first opening that does not converge within 200 iterations.
- **Line search.** With the sparse integration rule, $n_x = 60$, the iterations without safeguards (`nols_nx60`) enter an exact cycle after the peak, which no number of iterations can break. The line search (`ls_nx60`) breaks the cycle, but the criterion may still stagnate at a later opening.
- **Bisection.** With the bisection as well (`ref_nx60`) the analysis with $n_x = 60$ completes, the bisection being the safeguard of the coarse discretisations; on the converged discretisation (`ref`) it is not needed.
- None of the safeguards changes the equations or the convergence criterion: the converged loads are the same, only the path to them changes.